# Utility Placement Optimisation

**Learning outcome:** Apply utility placement optimisation through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `base`  
**Expected runtime:** under 2 minutes  
**Optional extras:** plot

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Where should two isothermal and two sensible hot and cold utility levels be placed at Process and Site hierarchy levels to minimize thermodynamic cost?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Prepare the placement study

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [1]:
from OpenPinch import PinchWorkspace

workspace = PinchWorkspace(
    source="chocolate_factory.json", project_name="Site"
)
problem = workspace.use_case("baseline")
baseline_input = problem.to_problem_json()
def assert_reversed_utility_pairs(utilities):
    by_name = {utility["name"]: utility for utility in utilities}
    for suffix in ("iso_1", "iso_2", "sensible_1", "sensible_2"):
        hot = by_name[f"hot_{suffix}"]
        cold = by_name[f"cold_{suffix}"]
        assert abs(cold["t_supply"]["value"] - hot["t_target"]["value"]) < 1e-9
        assert abs(cold["t_target"]["value"] - hot["t_supply"]["value"]) < 1e-9

search_options = {
    "iteration_limit": 1,
    "evaluation_limit": 100,
    "candidate_limit": 2,
    "run_count": 1,
}
process_maximum_duties = {
    f"hot_{suffix}": 20.0
    for suffix in ("iso_1", "iso_2", "sensible_1", "sensible_2")
}

## Step 2: Optimize a Process Zone and build its standard GCC

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [2]:
process_case = problem.target.utility_placement(
    isothermal=2,
    sensible=2,
    zone="Almond",
    period_ids=("0",),
    maximum_duties=process_maximum_duties,
    options=search_options,
)
process_evidence = process_case.utility_placement_result
process_objective = process_evidence.best.aggregate_objective
process_fallback_penalty = process_evidence.best.fallback_penalty
process_utilities = process_case.to_problem_json()["utilities"]
process_named_utilities = [
    utility
    for utility in process_utilities
    if utility["name"] not in {"HU", "CU"}
]
assert len(process_named_utilities) == 8
assert_reversed_utility_pairs(process_utilities)
assert process_fallback_penalty.value > 0.0
assert any(utility["name"] == "HU" for utility in process_utilities)
for utility in process_named_utilities:
    maximum = process_maximum_duties.get(utility["name"])
    if maximum is not None:
        assert utility["maximum_heat_flow"] == {
            "value": maximum, "unit": "kW"
        }
process_hot_targets = [
    utility["t_target"]["value"]
    for utility in process_utilities
    if utility["type"] == "Hot"
]
process_cold_supplies = [
    utility["t_supply"]["value"]
    for utility in process_utilities
    if utility["type"] == "Cold"
]
# Reject the former edge-clustered result before plotting it.
assert min(process_hot_targets) < 75.0
assert max(process_cold_supplies) > 15.0
process_case = workspace.add(
    process_case,
    name="optimized_process_utilities",
    activate=False,
)
process_case.target.direct_heat_integration(
    zone="Almond", period_id="0"
)
process_zone = process_case.master_zone.get_subzone("Almond")
assert all(
    utility.heat_flow.value <= process_maximum_duties[utility.name] + 1e-9
    for utility in process_zone.hot_utilities
    if utility.name in process_maximum_duties
)
assert process_zone.hot_utilities.get_stream_by_name("HU").heat_flow.value > 0.0
process_summary = process_case.summary_frame()
process_gcc = process_case.plot.grand_composite_curve(
    zone_name="Almond"
)

## Step 3: Optimize the Site and build its standard Total Site Profile

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [3]:
site_case = problem.target.utility_placement(
    isothermal=2,
    sensible=2,
    period_ids=("0",),
    options=search_options,
)
site_evidence = site_case.utility_placement_result
site_objective = site_evidence.best.aggregate_objective
site_utilities = site_case.to_problem_json()["utilities"]
assert len(site_utilities) == 8
assert_reversed_utility_pairs(site_utilities)
site_hot_targets = [
    utility["t_target"]["value"]
    for utility in site_utilities
    if utility["type"] == "Hot"
]
site_cold_supplies = [
    utility["t_supply"]["value"]
    for utility in site_utilities
    if utility["type"] == "Cold"
]
assert min(site_hot_targets) < 80.0
assert max(site_cold_supplies) > 15.0
site_case = workspace.add(
    site_case,
    name="optimized_site_utilities",
    activate=False,
)
assert workspace.use_case("baseline").to_problem_json() == baseline_input
site_case.target.total_site_heat_integration(period_id="0")
site_summary = site_case.summary_frame()
site_tsp = site_case.plot.total_site_profiles()

## Review the result

Review both optimized cases exactly like normal cases: compare the Process utilities on the standard GCC with the Site utilities on the standard Total Site Profile, and retain each placement result for engineering review.

In [4]:
from IPython.display import display

display(process_objective)
display(process_fallback_penalty)
display(process_summary)
display(process_gcc)
display(site_objective)
display(site_summary)
display(site_tsp)

QuantityValue(value=0.05984815679388933, unit='kW/K')

QuantityValue(value=0.33603087814600113, unit='dimensionless')

,Scope,Zone Type,Integration Type,Target Method,Period ID,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,Site/Almond,Process Zone,Process,Heat Exchange,0,139.22 kW,12.14 kW,51.35 kW,33.00 degC,33.00 degC,"HU: 79.22 kW, hot_iso_1: 20.00 kW, hot_iso_2: ...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."


QuantityValue(value=0.6863313374698734, unit='kW/K')

,Scope,Zone Type,Integration Type,Target Method,Period ID,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch,Hot Utilities,Cold Utilities
0,Site,Site,Process,Heat Exchange,0,1082.54 kW,3401.69 kW,440.47 kW,33.00 degC,33.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
1,Site,Site,Utility,Heat Exchange,0,1330.81 kW,3649.96 kW,192.20 kW,50.19 degC,17.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 379.79 kW, hot_...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
2,Site/Almond,Process Zone,Process,Heat Exchange,0,139.22 kW,12.14 kW,51.35 kW,33.00 degC,33.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
3,Site/Blending,Process Zone,Process,Heat Exchange,0,1.16 kW,1.38 kW,0.79 kW,41.00 degC,41.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
4,Site/CIP,Process Zone,Process,Heat Exchange,0,13.88 kW,0.00 kW,0.00 kW,12.00 degC,12.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
5,Site/Cocoa,Process Zone,Process,Heat Exchange,0,139.92 kW,19.86 kW,88.46 kW,77.00 degC,33.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 139.92 kW, hot_...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
6,Site/Compressor,Process Zone,Process,Heat Exchange,0,0.00 kW,48.66 kW,0.00 kW,66.00 degC,66.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
7,Site/Conching,Process Zone,Process,Heat Exchange,0,2.88 kW,78.28 kW,27.14 kW,76.00 degC,76.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 2.88 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
8,Site/Debacterization,Process Zone,Process,Heat Exchange,0,22.07 kW,0.00 kW,0.00 kW,38.00 degC,38.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."
9,Site/Hazelnut,Process Zone,Process,Heat Exchange,0,222.16 kW,14.34 kW,0.00 kW,33.00 degC,33.00 degC,"hot_iso_1: 0.00 kW, hot_iso_2: 0.00 kW, hot_se...","cold_iso_1: 0.00 kW, cold_iso_2: 0.00 kW, cold..."


## Interpret the result

Compare the Process result against its direct GCC and the Site result against its Total Site Profile. Inspect physical entropy generation from the balanced composite curves: use CP * ln(T_out / T_in) in kelvin for sensible intervals and the signed Q / T limit for isothermal intervals. Confirm that no generated HU/CU fallback has positive duty.

## Adapt this template

Replace the sample with validated plant data, apply defensible temperature bounds, and increase the optimizer limits before making an engineering decision.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.